# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
# TODO: Load environment variables
load_dotenv(".env")

True

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.DefaultEmbeddingFunction()

In [7]:
# TODO: Create a collection
# Choose any name you want
name = "udaplay_game"

try:
    chroma_client.delete_collection(collection)
except Exception:
    pass

collection = chroma_client.create_collection(
    name =  name,
    embedding_function=embedding_fn
)

print("collection create successfuly")

InternalError: Collection [udaplay_game] already exists

### Add documents

In [ ]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext("game")[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [ ]:
# verification
print(collection.count())

print(type(embedding_fn))

print(collection.peek())

1
<class 'chromadb.api.types.DefaultEmbeddingFunction'>
{'ids': ['game'], 'embeddings': array([[-4.76837531e-02, -1.36362435e-02, -2.10048333e-02,
        -2.89396811e-02,  1.70631874e-02, -4.05941121e-02,
         1.59704424e-02,  1.06668295e-02, -4.79368269e-02,
        -4.16720845e-02, -3.25564817e-02, -1.52356299e-02,
         1.93946380e-02, -4.66961786e-02, -3.37735042e-02,
        -1.36072040e-02,  7.48894960e-02,  6.96155876e-02,
         7.18380809e-02, -2.69510262e-02, -1.07769845e-02,
        -7.88036510e-02, -3.30629535e-02,  3.80312279e-02,
        -5.10617010e-02,  2.15463750e-02, -4.50067371e-02,
         2.55762804e-02, -9.91723686e-02, -5.21281362e-02,
        -4.38891649e-02,  5.53386658e-03,  7.51421452e-02,
         2.68153455e-02, -1.96970981e-02, -9.63656977e-02,
        -3.82603519e-02, -1.22499391e-02, -9.61587131e-02,
         4.07789974e-03, -3.92131545e-02, -4.16621305e-02,
         1.49855046e-02,  1.75323170e-02,  1.10037275e-01,
         6.97339280e-03,  5

In [ ]:
query = "Which PlayStation racing games was released for PlayStation 1?"

result = collection.query(
    query_texts=[query],
    n_results =3
)

result

{'ids': [['game']],
 'embeddings': None,
 'documents': [['[PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'Platform': 'PlayStation 1',
    'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.',
    'Publisher': 'Sony Computer Entertainment',
    'YearOfRelease': 1997,
    'Genre': 'Racing',
    'Name': 'Gran Turismo'}]],
 'distances': [[0.8990564346313477]]}